# Transformer

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
# Cell 1: 데이터 로딩
import pandas as pd
import os

# 경로 설정
DATA_DIR = './data_filtering/filtered/'

# train 데이터 로드
train_data = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print("✅ train_data shape:", train_data.shape)
display(train_data.head())

# test 데이터 10개 로드
test_data_list = []
for i in range(10):
    test_path = os.path.join(DATA_DIR, f'TEST_{i:02d}.csv')
    df = pd.read_csv(test_path)
    test_data_list.append(df)
    print(f"✅ Loaded TEST_{i:02d}.csv | shape: {df.shape}")


✅ train_data shape: (102676, 6)


,date_ordinal,date,store_menu,store,menu,sales
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
3,738524,2023-01-04,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
4,738525,2023-01-05,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0


✅ Loaded TEST_00.csv | shape: (5404, 6)
✅ Loaded TEST_01.csv | shape: (5404, 6)
✅ Loaded TEST_02.csv | shape: (5404, 6)
✅ Loaded TEST_03.csv | shape: (5404, 6)
✅ Loaded TEST_04.csv | shape: (5404, 6)
✅ Loaded TEST_05.csv | shape: (5404, 6)
✅ Loaded TEST_06.csv | shape: (5404, 6)
✅ Loaded TEST_07.csv | shape: (5404, 6)
✅ Loaded TEST_08.csv | shape: (5404, 6)
✅ Loaded TEST_09.csv | shape: (5404, 6)


In [3]:
# Cell 2: 전처리 함수 정의 및 실행
import numpy as np

def preprocess_data(df):
    # 날짜형 변환
    df['date'] = pd.to_datetime(df['date'])
    df['date_ordinal'] = df['date'].map(pd.Timestamp.toordinal)

    # 'store_menu' 식별자 추가 (이미 있는 경우 생략 가능)
    if 'store_menu' not in df.columns:
        df['store_menu'] = df['store'] + "_" + df['menu']

    # 날짜 관련 파생 변수
    df['day_of_week'] = df['date'].dt.dayofweek      # 0=월 ~ 6=일
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    # 필요한 feature만 추출
    use_cols = [
        'date', 'date_ordinal', 'store_menu', 'sales',
        'day_of_week', 'is_weekend', 'month', 'day'
    ]
    return df[use_cols]

# train 데이터 전처리
train_data = preprocess_data(train_data)
display(train_data.head())

# test 데이터 전처리
for i in range(10):
    test_data_list[i] = preprocess_data(test_data_list[i])


,date,date_ordinal,store_menu,sales,day_of_week,is_weekend,month,day
0,2023-01-01,738521,느티나무 셀프BBQ_1인 수저세트,0,6,1,1,1
1,2023-01-02,738522,느티나무 셀프BBQ_1인 수저세트,0,0,0,1,2
2,2023-01-03,738523,느티나무 셀프BBQ_1인 수저세트,0,1,0,1,3
3,2023-01-04,738524,느티나무 셀프BBQ_1인 수저세트,0,2,0,1,4
4,2023-01-05,738525,느티나무 셀프BBQ_1인 수저세트,0,3,0,1,5


In [4]:
# Cell 3: Transformer 모델 정의
import torch
import torch.nn as nn

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1, output_len=7):
        super(TimeSeriesTransformer, self).__init__()
        
        self.model_type = 'Transformer'
        self.output_len = output_len
        self.d_model = d_model

        # 입력 임베딩: Linear -> Positional Encoding
        self.input_projection = nn.Linear(input_dim, d_model)

        # 포지셔널 인코딩
        self.positional_encoding = self._generate_positional_encoding(1000, d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 디코더 (예측용)
        self.decoder = nn.Linear(d_model, output_len)

    def forward(self, src):
        # src: [B, L, input_dim]
        src = self.input_projection(src)  # [B, L, d_model]

        pos_encoding = self.positional_encoding[:src.size(1), :].unsqueeze(0).to(src.device)  # [1, L, d_model]
        src = src + pos_encoding  # [B, L, d_model]

        src = src.permute(1, 0, 2)  # [L, B, d_model]
        output = self.transformer_encoder(src)  # [L, B, d_model]
        output = output.permute(1, 0, 2)  # [B, L, d_model]

        last_hidden = output[:, -1, :]  # [B, d_model]
        pred = self.decoder(last_hidden)  # [B, 7]
        return pred


    def _generate_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe  # shape: [max_len, d_model] ← [1, max_len, d_model] ❌ X



In [5]:
from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, df, input_len=28, output_len=7):
        self.input_len = input_len
        self.output_len = output_len
        self.data = []
        
        store_menus = df['store_menu'].unique()
        for sm in store_menus:
            df_sm = df[df['store_menu'] == sm].sort_values('date')
            values = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values  # shape: [T, F]
            for i in range(len(values) - input_len - output_len + 1):
                x = values[i:i+input_len]
                y = values[i+input_len:i+input_len+output_len, 0]  # sales만 target
                self.data.append((x, y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


def split_by_store_menu(df, split_ratio=0.9):
    train_df_list, val_df_list = [], []

    for sm in df['store_menu'].unique():
        df_sm = df[df['store_menu'] == sm].sort_values('date')
        split_idx = int(len(df_sm) * split_ratio)
        train_df_list.append(df_sm.iloc[:split_idx])
        val_df_list.append(df_sm.iloc[split_idx:])

    train_df = pd.concat(train_df_list)
    val_df = pd.concat(val_df_list)
    return train_df, val_df

# ✅ 수정된 split
train_data_sorted = train_data.sort_values('date')
train_split, val_split = split_by_store_menu(train_data_sorted)

# 시퀀스 생성
train_dataset = SequenceDataset(train_split)
val_dataset = SequenceDataset(val_split)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [6]:
# 첫 번째 샘플 (index 0) 확인
x, y = val_dataset[0]

print("Input (x) shape:", x.shape)
print("Input (x):", x)

print("Target (y) shape:", y.shape)
print("Target (y):", y)


Input (x) shape: torch.Size([28, 5])
Input (x): tensor([[ 4.,  1.,  0.,  4., 23.],
        [ 0.,  2.,  0.,  4., 24.],
        [19.,  3.,  0.,  4., 25.],
        [ 3.,  4.,  0.,  4., 26.],
        [13.,  5.,  1.,  4., 27.],
        [ 8.,  6.,  1.,  4., 28.],
        [ 0.,  0.,  0.,  4., 29.],
        [ 5.,  1.,  0.,  4., 30.],
        [ 4.,  2.,  0.,  5.,  1.],
        [ 4.,  3.,  0.,  5.,  2.],
        [ 7.,  4.,  0.,  5.,  3.],
        [37.,  5.,  1.,  5.,  4.],
        [ 7.,  6.,  1.,  5.,  5.],
        [ 4.,  0.,  0.,  5.,  6.],
        [ 9.,  1.,  0.,  5.,  7.],
        [ 0.,  2.,  0.,  5.,  8.],
        [ 0.,  3.,  0.,  5.,  9.],
        [ 4.,  4.,  0.,  5., 10.],
        [26.,  5.,  1.,  5., 11.],
        [ 2.,  6.,  1.,  5., 12.],
        [ 0.,  0.,  0.,  5., 13.],
        [ 4.,  1.,  0.,  5., 14.],
        [ 3.,  2.,  0.,  5., 15.],
        [ 3.,  3.,  0.,  5., 16.],
        [ 5.,  4.,  0.,  5., 17.],
        [ 8.,  5.,  1.,  5., 18.],
        [ 5.,  6.,  1.,  5., 19.],
       

In [7]:
# Cell 5: 모델 학습 루프 정의
import torch.optim as optim
from tqdm import tqdm

# 하이퍼파라미터
input_dim = 5            # sales + day_of_week + is_weekend + month + day
d_model = 64
nhead = 4
num_layers = 2
output_len = 7
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 초기화
model = TimeSeriesTransformer(input_dim, d_model, nhead, num_layers, output_len=output_len).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 학습 루프
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0
    for x_batch, y_batch in tqdm(train_loader, desc=f"[Epoch {epoch}] Training", leave=False):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        output = model(x_batch)  # shape: [B, 7]
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # 검증
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output = model(x_batch)
            loss = criterion(output, y_batch)
            val_loss += loss.item() * x_batch.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"✅ Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


/home/wonjun/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


✅ Epoch 01 | Train Loss: 1386.6146 | Val Loss: 368.6741


✅ Epoch 02 | Train Loss: 1062.1100 | Val Loss: 539.0800


✅ Epoch 03 | Train Loss: 1011.3196 | Val Loss: 425.0768


✅ Epoch 04 | Train Loss: 958.2355 | Val Loss: 379.1211


...

✅ Epoch 06 | Train Loss: 877.8613 | Val Loss: 402.5971


✅ Epoch 07 | Train Loss: 852.6935 | Val Loss: 376.0146


✅ Epoch 08 | Train Loss: 841.9284 | Val Loss: 393.3220


✅ Epoch 09 | Train Loss: 844.2305 | Val Loss: 411.0491


✅ Epoch 10 | Train Loss: 835.7698 | Val Loss: 393.7043


In [8]:
# Cell 6: 테스트 데이터 추론 함수
def predict_test_data(model, test_df, input_len=28, device='cpu'):
    model.eval()
    preds = {}
    
    store_menus = test_df['store_menu'].unique()

    for sm in store_menus:
        df_sm = test_df[test_df['store_menu'] == sm].sort_values('date')
        x = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values[-input_len:]
        
        if x.shape[0] != input_len:
            # 입력 길이가 부족할 경우 패스
            continue

        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 28, 5]
        with torch.no_grad():
            pred = model(x_tensor)  # [1, 7]
        preds[sm] = pred.squeeze(0).cpu().numpy()
    
    return preds  # Dict[str, np.ndarray]

# 모든 test 파일에 대해 예측 수행
all_test_preds = {}

for i in range(10):
    test_df = test_data_list[i]
    pred_dict = predict_test_data(model, test_df, device=device)
    all_test_preds[f'TEST_{i:02d}'] = pred_dict
    print(f"✅ Predicted TEST_{i:02d}: {len(pred_dict)} store_menus")


✅ Predicted TEST_00: 193 store_menus
✅ Predicted TEST_01: 193 store_menus
✅ Predicted TEST_02: 193 store_menus
✅ Predicted TEST_03: 193 store_menus
✅ Predicted TEST_04: 193 store_menus
✅ Predicted TEST_05: 193 store_menus
✅ Predicted TEST_06: 193 store_menus
✅ Predicted TEST_07: 193 store_menus
✅ Predicted TEST_08: 193 store_menus
✅ Predicted TEST_09: 193 store_menus


In [12]:
# Cell 7: 음수 예측값 → 0으로 클리핑
for test_key in all_test_preds:
    for sm_key in all_test_preds[test_key]:
        all_test_preds[test_key][sm_key] = np.clip(all_test_preds[test_key][sm_key], a_min=0, a_max=None)

print("✅ 모든 음수 예측값을 0으로 변환 완료")


✅ 모든 음수 예측값을 0으로 변환 완료


In [15]:
# Cell 8: 예측 결과를 sample_submission에 채워넣기
import pandas as pd

# sample submission 불러오기
submission = pd.read_csv('./result/sample_submission.csv', index_col=0)
print("✅ 불러온 submission shape:", submission.shape)

# 예측값 채워넣기
for test_key in all_test_preds:  # e.g., TEST_00
    for store_menu in all_test_preds[test_key]:
        preds = all_test_preds[test_key][store_menu]  # shape: (7,)
        for day_offset in range(7):
            row_idx = f"{test_key}+{day_offset+1}일"
            if store_menu in submission.columns:
                submission.at[row_idx, store_menu] = preds[day_offset]

# 최종 확인
display(submission.head(10))


✅ 불러온 submission shape: (70, 193)


/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12.83161735534668' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '36.86190414428711' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_4624/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.995957374572754' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykerne

,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,느티나무 셀프BBQ_쌈장,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
영업일자,,,,,,,,,,,,,,,,,,,,,
TEST_00+1일,12.831617,36.861904,6.995957,0.0,0,0,17.731636,0.0,0.0,0,...,5.006301,24.225460,18.846436,6.370508,78.592491,53.861362,0.000000,51.061302,15.576219,29.443884
TEST_00+2일,13.496630,36.086525,7.998787,0.0,0,0,18.109169,0.0,0.0,0,...,6.123479,24.216896,19.158089,7.409432,75.174896,52.025406,0.636731,49.400894,16.080816,29.120462
TEST_00+3일,13.229271,35.123207,7.895943,0.0,0,0,17.704159,0.0,0.0,0,...,6.076081,23.623928,18.720018,7.324275,72.928429,50.544617,0.751566,48.005913,15.736056,28.374762
TEST_00+4일,13.465848,35.062481,8.203261,0.0,0,0,17.879202,0.0,0.0,0,...,6.407313,23.720442,18.882572,7.638919,72.363037,50.280239,1.150665,47.775265,15.938779,28.407089
TEST_00+5일,13.189157,34.689110,7.956846,0.0,0,0,17.577477,0.0,0.0,0,...,6.171802,23.391935,18.577230,7.395801,71.902069,49.864113,0.948940,47.365784,15.648553,28.059584
TEST_00+6일,12.580785,34.369061,7.285683,0.0,0,0,17.025633,0.0,0.0,0,...,5.479823,22.914406,18.036619,6.718202,72.163528,49.770473,0.200868,47.234344,15.070693,27.645222
TEST_00+7일,12.643683,34.906765,7.232740,0.0,0,0,17.182570,0.0,0.0,0,...,5.387357,23.204128,18.219398,6.652845,73.538513,50.653332,0.000000,48.059814,15.188267,28.039301
TEST_01+1일,4.675816,3.453650,9.396981,0.0,0,0,0.000000,0.0,0.0,0,...,0.000000,8.211305,5.971970,0.000000,37.938831,37.359467,0.000000,36.846214,0.000000,31.185034
TEST_01+2일,5.812617,4.659813,10.262018,0.0,0,0,0.000000,0.0,0.0,0,...,0.000000,9.144945,7.034339,0.000000,37.095978,36.552353,0.000000,36.070526,1.124654,30.756645


In [16]:
# 저장
submission.to_csv('./result/Vanilla_transformer.csv')
print("✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer.csv")


✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer.csv
